# Chuẩn bị bộ instruction tiếng Việt cho SFT

Fine-tune hỏng phần lớn vì dữ liệu, không vì siêu tham số. Bài này đi qua các bước một kỹ sư làm với bộ instruction
**trước** khi train: đọc giấy phép, chuẩn hoá Unicode, đưa về định dạng hội thoại, khử trùng lặp chính xác và gần đúng
(MinHash + LSH), lọc theo độ dài token, chia train/validation **theo nhóm** để bản gần trùng không rơi vào cả hai phía,
và đo nhiễm bẩn (contamination) giữa tập train và tập đánh giá bằng n-gram. Dữ liệu là `OpenOrca-Viet`
(121 178 mẫu instruction tiếng Việt). Kết quả ghi ra hai tệp mà bài Intermediate 02 dùng để SFT.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, in, ghi tệp; test bỏ qua chúng.
- Chỉ cần CPU. Lần đầu tải tokenizer của Qwen2.5 (vài MB).
- Chấm: `uv run pytest tracks/ai-engineer/modules/14-fine-tuning-llm/exercises/intermediate/01-instruction-data -q`

In [ ]:
# explore: tìm gốc repo, tải OpenOrca-Viet nếu chưa có, in giấy phép trong registry
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import DATASETS, ensure

ORCA = ensure("openorca-viet")
CACHE = ORCA / "cache"
CACHE.mkdir(exist_ok=True)
info = DATASETS["openorca-viet"]
print(ORCA)
print("giấy phép:", info.license)
print("nguồn:", info.homepage)

In [ ]:
import hashlib
import re
import unicodedata

import numpy as np

## 1. Chuẩn hoá và định dạng hội thoại

- `normalize_text(text) -> str`: NFC; `\r\n` và `\r` thành `\n`; trong mỗi dòng, gộp mọi chuỗi khoảng trắng ngang
  (space, tab, NBSP `\u00a0`) thành một space rồi strip hai đầu dòng; ba dòng trống liền nhau trở lên thành đúng một dòng
  trống (`\n\n`); strip cả chuỗi.
- `to_messages(row) -> list[dict]`: `row` có `instruction` (system prompt), `input`, `output`. Trả `system` (nếu có),
  `user`, `assistant`, nội dung đã `normalize_text`. **Bỏ** message `system` khi system prompt sau chuẩn hoá ngắn hơn 10
  ký tự hoặc không có ký tự chữ/số nào (bộ này có hơn mười nghìn mẫu mà system prompt chỉ là một dấu `"`, lỗi khi dịch).

In [ ]:
def normalize_text(text: str) -> str:
    # TODO
    raise NotImplementedError


def to_messages(row: dict) -> list[dict]:
    # TODO
    raise NotImplementedError

## 2. Khử trùng lặp chính xác và gần đúng

- `dedup_key(messages) -> str`: khoá trùng lặp của một mẫu: nội dung `user` và nội dung `assistant`, mỗi phần chữ
  thường và mọi chuỗi khoảng trắng (kể cả xuống dòng) gộp thành một space, nối hai phần bằng ký tự `\x1f`, rồi SHA-1
  (hex). System prompt **không**
  thuộc khoá: cùng câu hỏi, cùng câu trả lời mà khác system prompt vẫn là một mẫu.
- `exact_dedup(keys) -> list[int]`: chỉ số các mẫu giữ lại (lần xuất hiện đầu tiên của mỗi khoá), tăng dần.
- `minhash_signature(text, num_perm=64, ngram=5, seed=0) -> np.ndarray`: shingle là các chuỗi con `ngram` ký tự của
  văn bản đã chữ thường và gộp khoảng trắng (văn bản ngắn hơn `ngram` thì cả văn bản là một shingle). Hash mỗi shingle
  bằng `int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest(), "little") % P` với `P = 2**31 - 1`. Với `rng = np.random.default_rng(seed)`, lấy
  `a = rng.integers(1, P, num_perm, dtype=np.uint64)` rồi `b = rng.integers(0, P, num_perm, dtype=np.uint64)`; thành
  phần thứ `k` của chữ ký là `min((a[k] * h + b[k]) % P)` trên mọi shingle. Trả mảng `uint64` dài `num_perm`.
- `near_duplicate_groups(signatures, bands=16, threshold=0.8) -> np.ndarray`: LSH theo band: chia chữ ký thành `bands`
  đoạn liền nhau; trong mỗi band, mẫu nào có đoạn chữ ký trùng với mẫu **đầu tiên** đã vào cùng bucket thì là ứng viên;
  chỉ nối hai mẫu khi tỉ lệ thành phần chữ ký bằng nhau (ước lượng Jaccard) đạt `threshold`. Gộp nhóm bắc cầu
  (union-find). Trả mã nhóm cho mỗi mẫu: **chỉ số nhỏ nhất** trong nhóm.

In [ ]:
P = 2**31 - 1


def dedup_key(messages: list[dict]) -> str:
    # TODO
    raise NotImplementedError


def exact_dedup(keys: list[str]) -> list[int]:
    # TODO
    raise NotImplementedError


def minhash_signature(text: str, num_perm: int = 64, ngram: int = 5, seed: int = 0) -> np.ndarray:
    # TODO
    raise NotImplementedError


def near_duplicate_groups(signatures: np.ndarray, bands: int = 16, threshold: float = 0.8) -> np.ndarray:
    # TODO
    raise NotImplementedError

## 3. Chia theo nhóm và đo nhiễm bẩn

- `group_split(group_ids, val_frac, seed=0) -> np.ndarray`: mảng bool, `True` là validation. Quyết định theo **nhóm**,
  không theo mẫu: lấy SHA-1 của chuỗi `f"{seed}:{nhóm}"`, đọc 8 byte đầu thành số nguyên big-endian, chia cho `2**64`;
  nhóm vào validation khi giá trị đó nhỏ hơn `val_frac`. Cùng nhóm luôn cùng phía, không phụ thuộc thứ tự mẫu.
- `contamination(train_texts, eval_texts, n=13) -> np.ndarray`: với mỗi văn bản đánh giá, tỉ lệ word n-gram của nó
  (từ là `\w+` sau NFC và chữ thường) có mặt ở **bất kỳ** văn bản train nào. Văn bản ít hơn `n` từ cho 0.

In [ ]:
def group_split(group_ids, val_frac: float, seed: int = 0) -> np.ndarray:
    # TODO
    raise NotImplementedError


def contamination(train_texts: list[str], eval_texts: list[str], n: int = 13) -> np.ndarray:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc bộ dữ liệu, xem lỗi dữ liệu thật trước khi làm gì
import json
import time
from collections import Counter

rows = [json.loads(line) for line in (ORCA / "OpenOrca-Viet.jsonl").open(encoding="utf-8")]
raw = [r["instruction"] + r["input"] + r["output"] for r in rows]
systems = Counter(r["instruction"] for r in rows)
print(f"{len(rows)} mẫu, {len(systems)} system prompt khác nhau, "
      f"{sum(unicodedata.normalize('NFC', t) != t for t in raw)} mẫu không ở dạng NFC")
for s, c in systems.most_common(6):
    print(f"{c:6d}  {s[:90]!r}")

In [ ]:
# explore: định dạng hội thoại, khử trùng lặp chính xác
t0 = time.perf_counter()
msgs = [to_messages(r) for r in rows]
keep = exact_dedup([dedup_key(m) for m in msgs])
print(f"không còn system: {sum(m[0]['role'] != 'system' for m in msgs)} | sau khử trùng chính xác: {len(keep)} "
      f"(bỏ {len(rows) - len(keep)}) | {time.perf_counter() - t0:.0f} s")
print(msgs[2])

In [ ]:
# explore: độ dài token theo chat template của Qwen2.5 (lần đầu tải tokenizer)
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
t0 = time.perf_counter()
texts = [tok.apply_chat_template(msgs[i], tokenize=False) for i in keep]
lengths = np.array([len(x) for x in tok(texts, add_special_tokens=False)["input_ids"]])
print("token p10/p50/p90/p99/max:", np.percentile(lengths, [10, 50, 90, 99, 100]).astype(int),
      f"| {time.perf_counter() - t0:.0f} s")
MIN_TOK, MAX_TOK = 16, 512
print(f"trong [{MIN_TOK}, {MAX_TOK}] token: {int(((lengths >= MIN_TOK) & (lengths <= MAX_TOK)).sum())}")

In [ ]:
# explore: MinHash + LSH trên câu hỏi + câu trả lời (vài phút trên CPU; chữ ký được cache)
sig_text = [msgs[i][-2]["content"] + "\n" + msgs[i][-1]["content"] for i in keep]
sig_path = CACHE / "m14-minhash64.npy"
t0 = time.perf_counter()
if sig_path.exists() and len(np.load(sig_path)) == len(keep):
    sigs = np.load(sig_path)
else:
    sigs = np.stack([minhash_signature(t) for t in sig_text])
    np.save(sig_path, sigs)
groups = near_duplicate_groups(sigs, bands=16, threshold=0.8)
sizes = Counter(groups.tolist())
print(f"{len(sizes)} nhóm, {sum(v for v in sizes.values() if v > 1)} mẫu nằm trong nhóm có từ 2 mẫu, "
      f"nhóm lớn nhất {max(sizes.values())} | {time.perf_counter() - t0:.0f} s")
big = max(sizes, key=sizes.get)
for i in [k for k, g in enumerate(groups) if g == big][:3]:
    print(repr(sig_text[i][:160]))

In [ ]:
# explore: chia ngẫu nhiên và chia theo nhóm: bao nhiêu mẫu validation có bản gần trùng trong train
is_val_group = group_split(groups, 0.05, seed=0)
is_val_random = np.random.default_rng(0).random(len(keep)) < 0.05


def leaked(is_val):
    train_groups = set(groups[~is_val].tolist())
    return int(sum(g in train_groups for g in groups[is_val]))


for name, is_val in (("ngẫu nhiên", is_val_random), ("theo nhóm", is_val_group)):
    print(f"chia {name}: {int(is_val.sum())} mẫu val, {leaked(is_val)} mẫu val có bản gần trùng trong train")

In [ ]:
# explore: nhiễm bẩn 13-gram của 500 mẫu val với toàn bộ train, hai cách chia
rng = np.random.default_rng(0)
for name, is_val in (("ngẫu nhiên", is_val_random), ("theo nhóm", is_val_group)):
    sample = rng.choice(np.where(is_val)[0], size=500, replace=False)
    c = contamination([sig_text[i] for i in np.where(~is_val)[0]], [sig_text[i] for i in sample], n=13)
    print(f"chia {name}: {int((c >= 0.5).sum())}/500 mẫu val có >= 50% 13-gram nằm trong train, trung bình {c.mean():.3f}")

In [ ]:
# explore: một mẫu mỗi nhóm, lọc độ dài, ghi tệp cho bài 02 (10 000 mẫu train ngẫu nhiên, mọi mẫu val)
first_of_group = {}
for pos, g in enumerate(groups):
    first_of_group.setdefault(int(g), pos)
chosen = sorted(first_of_group.values())
ok = [p for p in chosen if MIN_TOK <= lengths[p] <= MAX_TOK]
train_pos = [p for p in ok if not is_val_group[p]]
val_pos = [p for p in ok if is_val_group[p]]
train_pos = sorted(np.random.default_rng(0).choice(train_pos, size=10_000, replace=False).tolist())
for name, positions in (("m14-train.jsonl", train_pos), ("m14-val.jsonl", val_pos)):
    with (CACHE / name).open("w", encoding="utf-8") as f:
        for p in positions:
            f.write(json.dumps({"id": int(keep[p]), "group": int(groups[p]), "n_tokens": int(lengths[p]),
                                "messages": msgs[keep[p]]}, ensure_ascii=False) + "\n")
print(f"một mẫu mỗi nhóm: {len(chosen)} | qua lọc độ dài: {len(ok)} | ghi {len(train_pos)} train, {len(val_pos)} val")

## Đọc kết quả

- Bộ dữ liệu ghi giấy phép gì, và vì sao "Apache-2.0" trên thẻ chưa đủ để dùng thương mại yên tâm?
- Khử trùng chính xác bỏ được bao nhiêu mẫu, MinHash gộp thêm bao nhiêu? Nhìn ba mẫu của nhóm lớn nhất: đó là trùng
  thật hay chỉ chung khuôn câu hỏi? Ngưỡng 0.8 có hợp lý không?
- Chia ngẫu nhiên để lọt bao nhiêu mẫu val có bản gần trùng trong train? Loss validation đo trên tập đó sẽ lạc quan
  hơn thực tế theo hướng nào?
- Phân phối độ dài token: chọn `MAX_TOK` 512 bỏ đi bao nhiêu phần trăm? Bỏ mẫu dài hay cắt cụt, cái nào an toàn hơn cho SFT?

Sau đó làm phần trắc nghiệm của bài trong app.